# Experimental Model Comparison — All Models on Kestrel

Compare all production and experimental runtime prediction models on the same
Kestrel data with the same rolling windows.

**Production models:** Baseline, XGBoost, Random Forest, MLP, TF-IDF kNN

**Experimental models:** LightGBM, XGBoost DART, XGBoost Tuned

**Dataset:** NLR Kestrel, benchmark window 2025-03-29 to 2025-06-26

**Related:** Issue [#124](https://github.com/NatLabRockies/hpc-oda-commons/issues/124)

## 1. Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Production models
from hpc_oda_commons.models.job_runtime_baseline.model import JobRuntimeBaselineModel
from hpc_oda_commons.models.job_runtime_xgboost.model import (
    JobRuntimeXGBoostConfig, JobRuntimeXGBoostModel,
)
from hpc_oda_commons.models.job_runtime_random_forest.model import (
    JobRuntimeRandomForestConfig, JobRuntimeRandomForestModel,
)
from hpc_oda_commons.models.job_runtime_mlp.model import (
    JobRuntimeMlpConfig, JobRuntimeMlpModel,
)
from hpc_oda_commons.models.job_runtime_tfidf_knn.model import (
    JobRuntimeTfidfKnnConfig, JobRuntimeTfidfKnnModel,
)

# Experimental models
from hpc_oda_commons.models.experimental.lightgbm_model import (
    ExperimentalLightGBMConfig, ExperimentalLightGBMModel,
)
from hpc_oda_commons.models.experimental.xgboost_dart_model import (
    ExperimentalXGBoostDartConfig, ExperimentalXGBoostDartModel,
)
from hpc_oda_commons.models.experimental.xgboost_tuned_model import (
    ExperimentalXGBoostTunedConfig, ExperimentalXGBoostTunedModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

In [ ]:
# Load and slice to benchmark window
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 3, 29, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()

SAMPLE_SIZE = 20_000
full_size = len(df)
if len(df) > SAMPLE_SIZE:
    df = df.tail(SAMPLE_SIZE).copy()

rows = df.to_dict('records')
print(f'Full window: {full_size:,} rows')
print(f'Using: {len(rows):,} rows (last {SAMPLE_SIZE:,} from window)')

## 2. Define models and shared config

All models use the same rolling evaluation parameters so results are comparable.

In [ ]:
# Shared rolling eval parameters
SHARED = dict(
    n_windows=4,
    test_window_hours=6,
    training_lookback_days=30,
    max_svd_components=16,
    target_max_one_hot_width=128,
    random_state=42,
)

# All models to compare
MODELS = {
    # Production models
    'XGBoost (production)': (
        JobRuntimeXGBoostModel,
        JobRuntimeXGBoostConfig(**SHARED),
    ),
    'Random Forest': (
        JobRuntimeRandomForestModel,
        JobRuntimeRandomForestConfig(**SHARED, n_estimators=100, max_depth=16),
    ),
    'MLP': (
        JobRuntimeMlpModel,
        JobRuntimeMlpConfig(**SHARED, hidden_layer_sizes=(128, 64), max_iter=200),
    ),
    'TF-IDF kNN': (
        JobRuntimeTfidfKnnModel,
        JobRuntimeTfidfKnnConfig(
            n_windows=SHARED['n_windows'],
            test_window_hours=SHARED['test_window_hours'],
            training_lookback_days=SHARED['training_lookback_days'],
            k=5, n_hash_features=2**14,
        ),
    ),
    # Experimental models
    'LightGBM': (
        ExperimentalLightGBMModel,
        ExperimentalLightGBMConfig(**SHARED),
    ),
    'XGBoost DART': (
        ExperimentalXGBoostDartModel,
        ExperimentalXGBoostDartConfig(**SHARED, rate_drop=0.1, skip_drop=0.5),
    ),
    'XGBoost Tuned': (
        ExperimentalXGBoostTunedModel,
        ExperimentalXGBoostTunedConfig(
            n_windows=SHARED['n_windows'],
            test_window_hours=SHARED['test_window_hours'],
            training_lookback_days=SHARED['training_lookback_days'],
            max_svd_components=SHARED['max_svd_components'],
            target_max_one_hot_width=SHARED['target_max_one_hot_width'],
            random_state=SHARED['random_state'],
            n_estimators=200, max_depth=12, learning_rate=0.03,
        ),
    ),
}

print(f'Models to compare: {len(MODELS)}')
for name in MODELS:
    print(f'  - {name}')

## 3. Run all models

Each model is evaluated on the same data with the same rolling windows.

In [ ]:
results = {}

for name, (model_cls, config) in MODELS.items():
    print(f'\nRunning {name}...')
    try:
        model = model_cls(config)
        payload = model.evaluate(rows)
        mae = payload['mae']
        rmse = payload['rmse']
        scored = payload['summary']['rows_scored']
        results[name] = {'mae': mae, 'rmse': rmse, 'scored': scored}
        print(f'  MAE={mae:,.1f}s  RMSE={rmse:,.1f}s  scored={scored:,}')
    except Exception as e:
        print(f'  FAILED: {e}')
        results[name] = {'mae': None, 'rmse': None, 'scored': 0, 'error': str(e)}

# Also compute baseline (mean predictor) using the rolling baseline runner
print('\nRunning Baseline (mean predictor)...')
from hpc_oda_commons.benchmark.runner import run_rolling_baseline
split_params = {
    'method': 'rolling',
    'n_windows': SHARED['n_windows'],
    'test_window_hours': SHARED['test_window_hours'],
    'training_lookback_days': SHARED['training_lookback_days'],
}
metric_defs = [
    {'name': 'mae', 'target': 'runtime_seconds'},
    {'name': 'rmse', 'target': 'runtime_seconds'},
]
bl_metrics, bl_payload, _ = run_rolling_baseline(rows, split=split_params, metric_defs=metric_defs)
bl_scored = bl_payload['summary']['rows_scored']
results['Baseline (mean)'] = {'mae': bl_metrics['mae'], 'rmse': bl_metrics['rmse'], 'scored': bl_scored}
print(f'  MAE={bl_metrics["mae"]:,.1f}s  RMSE={bl_metrics["rmse"]:,.1f}s  scored={bl_scored:,}')

## 4. Results Table

In [ ]:
# Sort by MAE (best first)
valid = {k: v for k, v in results.items() if v['mae'] is not None}
sorted_results = sorted(valid.items(), key=lambda x: x[1]['mae'])

print('=' * 75)
print('MODEL COMPARISON — NLR KESTREL')
print('=' * 75)
print(f'\n{"Rank":<5} {"Model":<25} {"MAE":>10} {"RMSE":>10} {"Scored":>8} {"Type"}')
print('-' * 75)

best_mae = sorted_results[0][1]['mae']
production_models = {'Baseline (mean)', 'XGBoost (production)', 'Random Forest', 'MLP', 'TF-IDF kNN'}

for rank, (name, r) in enumerate(sorted_results, 1):
    model_type = 'production' if name in production_models else 'experimental'
    vs_best = f'(+{(r["mae"] - best_mae) / best_mae * 100:.1f}%)' if rank > 1 else '(best)'
    print(f'{rank:<5} {name:<25} {r["mae"]:>10,.1f}s {r["rmse"]:>10,.1f}s {r["scored"]:>8,} {model_type} {vs_best}')

# Check for failed models
failed = {k: v for k, v in results.items() if v['mae'] is None}
if failed:
    print(f'\nFailed models:')
    for name, r in failed.items():
        print(f'  {name}: {r.get("error", "unknown error")}')

## 5. Bar Chart

In [ ]:
names = [name for name, _ in sorted_results]
maes = [r['mae'] for _, r in sorted_results]

# Color: blue for production, green for experimental
colors = ['steelblue' if n in production_models else 'seagreen' for n in names]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(names[::-1], maes[::-1], color=colors[::-1])

for bar, mae in zip(bars, maes[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{mae:,.0f}s', va='center', fontsize=10)

ax.set_xlabel('MAE (seconds, lower = better)')
ax.set_title('Model Comparison — NLR Kestrel\n(blue = production, green = experimental)')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='steelblue', label='Production'),
    Patch(color='seagreen', label='Experimental'),
], loc='lower right')

plt.tight_layout()
plt.show()

## 6. MAE vs RMSE Scatter

Models that are good on both MAE (average error) and RMSE (worst-case error)
appear in the bottom-left corner.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, r in valid.items():
    color = 'steelblue' if name in production_models else 'seagreen'
    marker = 'o' if name in production_models else 's'
    ax.scatter(r['mae'], r['rmse'], color=color, marker=marker, s=100, zorder=5)
    ax.annotate(name, (r['mae'], r['rmse']),
                xytext=(8, 4), textcoords='offset points', fontsize=9)

ax.set_xlabel('MAE (seconds)')
ax.set_ylabel('RMSE (seconds)')
ax.set_title('MAE vs RMSE — All Models\n(bottom-left = best on both metrics)')
ax.legend(handles=[
    Patch(color='steelblue', label='Production'),
    Patch(color='seagreen', label='Experimental'),
])
plt.tight_layout()
plt.show()

## 7. Conclusions

In [ ]:
print('CONCLUSIONS')
print('=' * 60)

best_name, best_r = sorted_results[0]
print(f'\nBest model: {best_name}')
print(f'  MAE:  {best_r["mae"]:,.1f}s')
print(f'  RMSE: {best_r["rmse"]:,.1f}s')

# Best production model
prod_results = [(n, r) for n, r in sorted_results if n in production_models]
best_prod_name, best_prod_r = prod_results[0]
print(f'\nBest production model: {best_prod_name}')
print(f'  MAE:  {best_prod_r["mae"]:,.1f}s')

# Best experimental model
exp_results = [(n, r) for n, r in sorted_results if n not in production_models]
if exp_results:
    best_exp_name, best_exp_r = exp_results[0]
    print(f'\nBest experimental model: {best_exp_name}')
    print(f'  MAE:  {best_exp_r["mae"]:,.1f}s')
    
    diff = (best_exp_r['mae'] - best_prod_r['mae']) / best_prod_r['mae'] * 100
    print(f'\nExperimental vs production best: {diff:+.1f}% MAE')
    if diff < -5:
        print('An experimental model significantly outperforms all production models.')
    elif diff > 5:
        print('Production models are better — experimental variants did not improve results.')
    else:
        print('Results are similar — experimental variants perform comparably to production.')

print(f'\nNote: this is a small-scale test ({SAMPLE_SIZE:,} rows, {SHARED["n_windows"]} windows).')
print('Full-scale benchmarks may show different results.')
print('Current models also use post-hoc features (issue #122) which may affect results.')